# **GPU Demo Part 2 - Decoupler-GPU: Accelerated Transcriptional Regulatory Analysis**  

## Check GPU Resources

Run only one GPU course at a time on this workshop VM.

1. **Save and close the previous kernel.** Save the previous course's notebook, then shut down its kernel in **Running Terminals and Kernels** in that course's JupyterLab. Closing a browser tab does not release GPU memory.
2. **Check GPU use.** Open **File → New → Terminal** and run `bootcamp-resources status`.
3. **Prepare this course.** If any BioNeMo models are running, run `bootcamp-resources stop --confirm`.
4. **Confirm readiness.** Run `bootcamp-resources status` again. Continue only when the BioNeMo models are stopped and `gpu_processes` is empty.

For Single-cell / Genomics, keep only one notebook kernel open at a time and use the kernel specified in the course guide.

For the complete instructions, see `00_START_HERE.txt` in the course root directory.

**Author:** [Severin Dicks](https://github.com/Intron7)
**Copyright** [scverse](https://scverse.org)

**Please run [GPU Demo Part 1 - Basic GPU E2E Workflow](01_scRNA_analysis_preprocessing.ipynb) before running this notebook as this notebook continues some analysis of that notebook's output.**

In this notebook, we explore the updated [**Decoupler**](https://decoupler-py.readthedocs.io/en/latest/) functions for inferring transcriptional regulatory activity, leveraging the **GPU-accelerated** implementation from **rapids-singlecell**.  

By running these analyses on GPUs, we achieve **faster computation** and **scalability** for large single-cell datasets. We will apply **Decoupler** methods to estimate **transcription factor (TF) activity** and **pathway activity**, utilizing curated regulatory networks such as **Dorothea** and **PROGENy**.


In [ ]:
import rapids_singlecell as rsc
import scanpy as sc
import rmm
import cupy as cp
import pandas as pd
import anndata as ad
import decoupler as dc
from rmm.allocators.cupy import rmm_cupy_allocator
rmm.reinitialize(
    managed_memory=False, # Allows oversubscription
    pool_allocator=True, # default is False
)
cp.cuda.set_allocator(rmm_cupy_allocator)

In [ ]:
adata = sc.read_h5ad("./h5/dli_decoupler.h5ad")

In [ ]:
adata = adata.raw.to_adata()

In [ ]:
rsc.get.anndata_to_GPU(adata)

### Retrieving Transcription Factor (TF) Regulons with Dorothea  

We start by retrieving the **Dorothea** gene regulatory network, which provides curated transcription factor (TF) target interactions.  
Dorothea assigns confidence levels (`A`, `B`, `C`, `D`, `E`), where `A` represents the highest confidence interactions.  
Here, we filter for **high-confidence** (`A`, `B`, `C`) interactions in human data:  

In [ ]:
collectri = dc.op.collectri(organism="human",license="commercial")

### Transcription Factor Activity Inference with ULM  

To infer transcription factor (TF) activity, we use the **Univariate Linear Model (ULM)** from **Decoupler**.  
ULM estimates the regulatory influence of TFs on gene expression by fitting a linear model for each TF-gene interaction.  

We apply **ULM** using the **Dorothea** regulatory network retrieved earlier. 

In [ ]:
%%time
rsc.dcg.ulm(adata, collectri, verbose=True, raw = False, pre_load=True)

In [ ]:
acts = dc.pp.get_obsm(adata=adata, key="score_ulm")

In [ ]:
sc.pl.embedding(acts,basis="X_umap_harmony", color=["cell_type", "GATA3"], cmap='RdBu_r',ncols=1)
sc.pl.violin(acts, keys=['GATA3'], groupby='cell_type', rotation=90, size = 0)


### Transcription Factor Activity Inference with MLM  

Next, we infer transcription factor (TF) activity using the **Multivariate Linear Model (MLM)** from **Decoupler**.  
Unlike **ULM**, which considers each TF independently, **MLM** accounts for multiple TFs simultaneously,  
providing a more comprehensive estimation of regulatory influences. 

In [ ]:
%%time
rsc.dcg.mlm(adata, collectri, verbose=True, raw = False, pre_load=True)

### Transcription Factor Activity Inference with AUCell  

Next, we infer transcription factor (TF) activity using **AUCell** from **Decoupler**.  
AUCell estimates TF activity based on the **enrichment of TF target genes within the top-expressed genes** in each cell.  
Unlike regression-based methods (**ULM**, **MLM**), AUCell is a **rank-based approach** that does not assume linear relationships.


In [ ]:
%%time
rsc.dcg.aucell(adata, collectri, verbose=True, raw = False, pre_load=True)

In [ ]:
acts = dc.pp.get_obsm(adata=adata, key="score_aucell")

In [ ]:
sc.pl.embedding(acts,basis="X_umap_harmony", color=["cell_type", "GATA3"], cmap='RdBu_r',ncols=1)
sc.pl.violin(acts, keys=['GATA3'], groupby='cell_type', rotation=90, size = 0)


In [ ]:
# These notebooks are very GPU memory intensive!
# In order to free up GPU memory, we'll kill this kernel prior to proceeding.  You will get a message.  This is expected.
# If you have a CUDA or an Out Of Memory (OOM) error, please kill all kernels to free up your GPU memory and try again!
# You can comment this out if you want to continue exploring the notebook.
# Please consult the README for more tips and tricks.

import IPython

IPython.Application.instance().kernel.do_shutdown(True)

## Save and Release GPU Resources

1. **Save and shut down.** Save this notebook, then shut down its kernel in **Running Terminals and Kernels**.
2. **Release resources.** In **File → New → Terminal**, run `bootcamp-resources status` to check that this course has released the GPU.
3. **Check before switching.** Run `bootcamp-resources status` and confirm that `gpu_processes` is empty before opening the next course's kernel.

If the notebook's cleanup cell restarts the kernel, shut down that restarted kernel in the Running panel before opening another notebook.

When returning to BioNeMo, follow **Check GPU Resources** at the beginning of its lesson and wait for all three models to become **healthy**.
